# Práctica Tema 4 — Pipeline de Inferencia (Tizón Tardío en Hojas de Papa)

**Objetivo:** construir y demostrar un flujo completo de inferencia, desde la carga del modelo hasta la predicción, para el clasificador `healthy` / `late_blight` (ResNet-18, transfer learning).

## Tareas del proyecto
- Cargar el modelo entrenado.
- Implementar preprocesamiento y postprocesamiento.
- Ejecutar inferencia **por lote** y **en tiempo real (simulado)**.

Este notebook reutiliza las funciones ya implementadas y probadas en `src/pipeline_inferencia.py` (pipeline operativo de producción), en vez de duplicar lógica: se importan directamente para demostrar cada etapa del flujo con evidencia ejecutada.


In [4]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pipeline_inferencia as pi

print("Raíz del proyecto:", PROJECT_ROOT)
print("Módulo de pipeline importado desde:", pi.__file__)


Raíz del proyecto: c:\Users\Jose_\OneDrive\Documentos\GitHub\ProyectoVisionComputadora
Módulo de pipeline importado desde: c:\Users\Jose_\OneDrive\Documentos\GitHub\ProyectoVisionComputadora\src\pipeline_inferencia.py


## 1. Cargar el modelo entrenado

Se selecciona el dispositivo disponible (CUDA / MPS / CPU), se leen las clases desde `config/clases.txt` y se reconstruye la arquitectura ResNet-18 antes de cargar los pesos (`model.pt` solo contiene el `state_dict`). `cargar_modelo()` además ejecuta una prueba de integridad (forward-pass dummy) para confirmar que el modelo responde correctamente.


In [5]:
dispositivo = pi.seleccionar_dispositivo("auto")
clases = pi.cargar_clases(PROJECT_ROOT / "config" / "clases.txt")
modelo = pi.cargar_modelo(PROJECT_ROOT / "models" / "model.pt", n_clases=len(clases), dispositivo=dispositivo)

print("Dispositivo seleccionado:", dispositivo)
print("Clases cargadas:", clases)
print("Modelo listo para inferencia:", modelo.eval() is not None)


2026-11-10 18:29:10,910 | INFO | Modelo cargado correctamente desde c:\Users\Jose_\OneDrive\Documentos\GitHub\ProyectoVisionComputadora\models\model.pt (dispositivo=cpu)
Dispositivo seleccionado: cpu
Clases cargadas: ['healthy', 'late_blight']
Modelo listo para inferencia: True


## 2. Preprocesamiento y postprocesamiento

**Preprocesamiento** (`preprocesar_imagen`): abre la imagen con PIL, la convierte a RGB, la redimensiona a 224×224, la convierte a tensor y la normaliza con la media/desviación estándar de ImageNet — idéntico al preprocesamiento usado en el entrenamiento.

**Postprocesamiento** (dentro de `ejecutar_inferencia`): aplica `softmax` a los logits del modelo para obtener probabilidades, toma la clase con mayor probabilidad (`argmax`) y la decodifica a su nombre (`healthy` / `late_blight`) usando `config/clases.txt`.


In [6]:
ruta_ejemplo = PROJECT_ROOT / "ejemplos" / "test1.jpg"

# Preprocesamiento
tensor, imagen_pil = pi.preprocesar_imagen(ruta_ejemplo)
print("Imagen original (tamaño):", imagen_pil.size)
print("Tensor preprocesado (forma):", tuple(tensor.shape))
print("Rango de valores tras normalizar:", (tensor.min().item(), tensor.max().item()))

# Inferencia + Postprocesamiento (softmax -> argmax -> clase)
resultado = pi.ejecutar_inferencia(modelo, tensor, clases, dispositivo)
print("\nResultado postprocesado:")
print(resultado)


Imagen original (tamaño): (750, 1000)
Tensor preprocesado (forma): (1, 3, 224, 224)
Rango de valores tras normalizar: (-1.804444432258606, 2.640000104904175)

Resultado postprocesado:
{'clase': 'late_blight', 'confianza_pct': 98.04, 'probabilidades': {'healthy': 1.96, 'late_blight': 98.04}, 'latencia_ms': 98.6}


## 3. Inferencia por lote

Se apilan varias imágenes ya preprocesadas en un único tensor `[N, 3, 224, 224]` y se ejecuta **un solo forward-pass** para todo el lote (`ejecutar_inferencia_lote`), en vez de llamar al modelo una vez por imagen. Esto es más eficiente, especialmente en GPU.


In [7]:
rutas_ejemplos = pi.resolver_entradas(imagen=None, directorio=str(PROJECT_ROOT / "ejemplos"))
print(f"Imágenes a procesar por lote: {[r.name for r in rutas_ejemplos]}")

tensores = []
for ruta in rutas_ejemplos:
    tensor, _ = pi.preprocesar_imagen(ruta)
    tensores.append(tensor)

resultados_lote = pi.ejecutar_inferencia_lote(modelo, tensores, clases, dispositivo)

print(f"\n{'Imagen':<14}{'Predicción':<14}{'Confianza %':<13}{'Latencia (ms)'}")
for ruta, resultado in zip(rutas_ejemplos, resultados_lote):
    print(f"{ruta.name:<14}{resultado['clase']:<14}{resultado['confianza_pct']:<13}{resultado['latencia_ms']}")


Imágenes a procesar por lote: ['test.jpg', 'test1.jpg', 'test2.jpg', 'test2.png']
2026-11-10 18:29:11,625 | INFO | Inferencia por lote: 4 imágenes en 226.06 ms (56.51 ms/imagen)

Imagen        Predicción    Confianza %  Latencia (ms)
test.jpg      healthy       61.62        56.51
test1.jpg     late_blight   98.04        56.51
test2.jpg     healthy       92.89        56.51
test2.png     late_blight   90.64        56.51


## 4. Inferencia en tiempo real (simulado)

En vez de esperar a tener todo el lote, se procesa **una imagen a la vez**, con un pequeño retardo entre "frames" (`ejecutar_tiempo_real_simulado`), simulando imágenes que llegan progresivamente (p. ej. una cámara o un stream en vivo). Cada resultado se muestra apenas está disponible, con marca de tiempo.


In [8]:
resultados_tiempo_real = pi.ejecutar_tiempo_real_simulado(
    modelo, rutas_ejemplos, clases, dispositivo, intervalo_segundos=0.5
)

print(f"\nTotal de frames procesados en tiempo real simulado: {len(resultados_tiempo_real)}")



Iniciando inferencia en tiempo real simulado (intervalo=0.5s entre frames)...
[18:29:11] Frame #001 | test.jpg                       -> healthy      (61.62%) | latencia=83.66 ms
2026-11-10 18:29:11,761 | INFO | TIEMPO_REAL | frame=1 | test.jpg -> healthy (61.62%, 83.66 ms)
[18:29:12] Frame #002 | test1.jpg                      -> late_blight  (98.04%) | latencia=78.38 ms
2026-11-10 18:29:12,363 | INFO | TIEMPO_REAL | frame=2 | test1.jpg -> late_blight (98.04%, 78.38 ms)
[18:29:13] Frame #003 | test2.jpg                      -> healthy      (92.89%) | latencia=128.99 ms
2026-11-10 18:29:13,029 | INFO | TIEMPO_REAL | frame=3 | test2.jpg -> healthy (92.89%, 128.99 ms)
[18:29:13] Frame #004 | test2.png                      -> late_blight  (90.64%) | latencia=90.48 ms
2026-11-10 18:29:13,666 | INFO | TIEMPO_REAL | frame=4 | test2.png -> late_blight (90.64%, 90.48 ms)

Total de frames procesados en tiempo real simulado: 4


## 5. Persistencia de resultados y conclusión

Ambos modos (`lote` y `tiempo-real`) también están disponibles como CLI en `src/pipeline_inferencia.py`:

```bash
python src/pipeline_inferencia.py --directorio ejemplos/ --modo lote --salida outputs/resultados_lote.json
python src/pipeline_inferencia.py --directorio ejemplos/ --modo tiempo-real --intervalo 1.0 --salida outputs/resultados_tiempo_real.json
```

**Resumen de lo demostrado en esta práctica:**
1. ✅ Carga del modelo entrenado (`cargar_modelo`), con reconstrucción de arquitectura y prueba de integridad.
2. ✅ Preprocesamiento (`preprocesar_imagen`) idéntico al del entrenamiento y postprocesamiento (softmax + argmax + decodificación de clase).
3. ✅ Inferencia **por lote** (`ejecutar_inferencia_lote`): un solo forward-pass para varias imágenes.
4. ✅ Inferencia **en tiempo real simulada** (`ejecutar_tiempo_real_simulado`): una imagen a la vez, con retardo entre frames y marca de tiempo.

El flujo completo de inferencia, desde la carga del modelo hasta la predicción, queda construido y verificado con evidencia ejecutada en este notebook.
